In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
h_big = 5
w_big = 5

h_small = 3
w_small = 3

avg_len = 0.4

In [ ]:
ipu, points, segment_edges, m, marker= periodic_unit_helper.get_boundary_aligned_dashline(w_small, w_big, h_small, h_big, avg_len)

In [ ]:
visualization.plot_line_segments(points, segment_edges)

In [ ]:
finalMarkers = np.where(np.array(marker) == 1)[0]

In [ ]:
n_hori_copy = 0
n_vert_copy = 0

In [ ]:
for i in range(n_hori_copy):
    m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)

for i in range(n_vert_copy):
    m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)

In [ ]:
visualization.plot_2d_mesh(m, pointList=finalMarkers, width=5, height=5)

In [ ]:
len(m.vertices())

In [ ]:
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

In [ ]:
fuse_boundary = False

In [ ]:
# Fuse boundary
if fuse_boundary:
    bbox = igl.bounding_box(m.vertices())

    max_x = max(bbox[0][:, 0])
    min_x = min(bbox[0][:, 0])
    max_y = max(bbox[0][:, 1])
    min_y = min(bbox[0][:, 1])

    vxs = m.vertices()
    for i, vx in enumerate(m.vertices()):
        if np.abs(vx[0] - max_x) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[0] - min_x) < 1e-6:
            fusedVtx[i] = True    
        if np.abs(vx[1] - max_y) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[1] - min_y) < 1e-6:
            fusedVtx[i] = True    

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
viewer.update()



In [ ]:
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])



In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
fixedVars, hessianShift = [ipu.numVars() - 2], 1e-6
# fixedVars, hessianShift = [], 1e-6

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = 3

In [ ]:
new_vars = ipu.getVars() + np.random.random(ipu.numVars()) * 1e-2
new_vars[-2] = 0

In [ ]:
ipu.setVars(new_vars)

In [ ]:
benchmark.reset()

opts.niter = 1000
opts.gradTol = 1e-7
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
optimizer = inflation.get_inflation_optimizer(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
cr = optimizer.optimize()
benchmark.report()

In [ ]:
ipu.visualizationTilePower = 0

In [ ]:
ipu.get_alpha()

In [ ]:
ipu.get_kappa()

In [ ]:
za_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx)

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
fixedVars, hessianShift = [za_ipu.numVars() - 2, 3, 4], 1e-6
# fixedVars, hessianShift = [za_ipu.numVars() - 2], 1e-6
# fixedVars, hessianShift = [], 1e-6

In [ ]:
benchmark.reset()

opts.niter = 1000
opts.gradTol = 1e-5
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(za_ipu.ipu.sheet)[:, 0])
az_optimizer = inflation.get_inflation_optimizer(za_ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
cr = az_optimizer.optimize()
benchmark.report()

In [ ]:
benchmark.reset()
stiffness, sampled_alphas = visualize_sampled_bending_stiffness(za_ipu, 100, az_optimizer, filename = "stiffness_shifted_dashline_{}_tessellation_{}_{}.png".format(0, n_hori_copy, n_vert_copy), hessianShift = 0, useFixedVars=True)
benchmark.report()

In [ ]:
stiffness